# LDA Comparison

Three stages. (A) Rebuild the CRS from the published keywords and export the backbone partition (408 concepts, 7 communities). (B) Fit LDA with scikit-learn on title and abstract of the 52,922 matched documents (K in {5, 10, 15, 20, 30, 40, 50}; unigrams and bigrams, min_df 10, max_df 0.8, 30,000 features; batch, 50 iterations, seed 42), select K by mean NPMI, assign each document to its dominant topic and to its majority CRS community, and measure NMI and ARI. (C) Community summaries, CRS x LDA cross-tabulation, inter-community edges and bridge concepts (Section IV-D3, Table 9).

Inputs: `EID_KEYWORDS.xlsx`, `data/insumo_row_to_eid.csv`, and the private `corpus_insumo_DEFINITIVO.csv` from `FTTS_PRIVATE_DIR`. Outputs in `results/e8_topic_modeling/` (fitted models are gitignored). Gates: the CRS must reproduce the published graph; the matched set must be the 52,922 EIDs of the archived assignment file; 30,000 features.

The matched set is the main block of the record file (rows 0 to 52,922); the 207 rows appended after it are a re-extraction pass (duplicates and rows without a published output). The co-author's fitting script was not archived; the fit follows `lda_config.json`.

LDA is seeded but platform dependent: with the same document-term matrix the fit converges to a different local optimum on another BLAS or CPU architecture, and NPMI, NMI and ARI move in the second decimal. Table 9 quotes the run made on an 8-core Apple silicon laptop, kept in `results/e8_topic_modeling/paper_run_macos/`. The CRS side is identical on every machine.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"          # four notebooks run concurrently on this machine
os.environ["HF_HUB_OFFLINE"] = "1"           # the embedding model must already be cached
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
sys.path.insert(0, "scripts")

import ast
import json
import platform
import time
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

import common as C                # PRIVATE_DIR, parse_record, load_alignment
import crs_reference as ref       # verbatim graph functions of notebooks 2, 3 and 5
import e3_scalability as e3       # load_corpus, metrics, full_reference_gate, constants

OUT = Path("results/e8_topic_modeling")
OUT.mkdir(parents=True, exist_ok=True)
PRIVATE_RECORDS = C.PRIVATE_DIR / "corpus_insumo_DEFINITIVO.csv"
ALIGNMENT = Path("data/insumo_row_to_eid.csv")
KEYWORDS = Path("EID_KEYWORDS.xlsx")
RECORD_MAIN_BLOCK_LAST_ROW = 52922     # rows 0..52922 of the record file; rows 52923..53129 are the appended re-extraction pass

KS = [5, 10, 15, 20, 30, 40, 50]
LDA = {"max_features": 30000, "min_df": 10, "max_df": 0.8, "ngram_range": (1, 2), "stop_words": "english",
       "max_iter": 50, "learning_method": "batch", "random_state": 42, "top_words_for_npmi": 10}
MAIN_COMMUNITIES = [0, 1, 2, 4]
TOP_N_CONCEPTS, TOP_N_TOPICS, TOP_N_BRIDGES = 15, 5, 20

if not PRIVATE_RECORDS.exists():
    raise FileNotFoundError(f"Private record file not found: {PRIVATE_RECORDS}. Set FTTS_PRIVATE_DIR to the directory "
                            "that holds corpus_insumo_DEFINITIVO.csv (Scopus records, Elsevier licence, not redistributed).")

# Results present before this execution (the committed run) are kept in memory for the comparison at the end.
ARCHIVED_FILES = ["crs_keyword_communities.csv", "crs_backbone_edges.csv", "crs_community_sizes.csv",
                  "lda_model_selection.csv", "lda_topics_all_k.csv", "lda_crs_comparison.csv",
                  "crs_document_community_distribution.csv", "crs_lda50_community_summary.csv",
                  "crs_intercommunity_edges.csv", "crs_intercommunity_pair_summary.csv", "crs_bridge_concepts.csv"]
ARCHIVED = {name: pd.read_csv(OUT / name) for name in ARCHIVED_FILES if (OUT / name).exists()}
ARCHIVED_ASSIGNMENTS = (pd.read_csv(OUT / "crs_document_assignments.csv")
                        if (OUT / "crs_document_assignments.csv").exists() else None)
ARCHIVED_CONFIG = json.loads((OUT / "lda_config.json").read_text()) if (OUT / "lda_config.json").exists() else None
print(f"Archived result files kept for the final comparison: {len(ARCHIVED)} tables"
      f"{'' if ARCHIVED_ASSIGNMENTS is None else ' + document assignments'}{'' if ARCHIVED_CONFIG is None else ' + lda_config.json'}")

torch.set_num_threads(4)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print("Private records:", PRIVATE_RECORDS)
print("Platform:", platform.platform(), "| cpu_count:", os.cpu_count(), "| torch threads:", torch.get_num_threads())
print("K values:", KS, "| LDA:", LDA)

/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Archived result files kept for the final comparison: 11 tables + document assignments + lda_config.json
Private records: /home/mat/academic-writing/papers/from_text_to_structure/corpus_insumo_DEFINITIVO.csv
Platform: Linux-7.0.0-31-generic-x86_64-with-glibc2.39 | cpu_count: 16 | torch threads: 4
K values: [5, 10, 15, 20, 30, 40, 50] | LDA: {'max_features': 30000, 'min_df': 10, 'max_df': 0.8, 'ngram_range': (1, 2), 'stop_words': 'english', 'max_iter': 50, 'learning_method': 'batch', 'random_state': 42, 'top_words_for_npmi': 10}


In [2]:
# ============================================================
# STAGE A: REBUILD THE CRS, PASS THE GATE, EXPORT THE BACKBONE PARTITION
# ============================================================
stage_started = time.perf_counter()
corpus, corpus_audit = e3.load_corpus()
print("Analysable documents:", len(corpus))
assert len(corpus) == 52946, f"expected 52946 documents, got {len(corpus)}"
keyword_lists = corpus["keywords"].tolist()

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
print("Model:", e3.MODEL, "| revision:", e3.REVISION[:12], "| tau:", e3.TAU, "| backbone w >=", e3.BACKBONE, "| Louvain seed:", e3.LOUVAIN_SEED)

t0 = time.perf_counter()
model = SentenceTransformer(e3.MODEL, revision=e3.REVISION, device="cpu", local_files_only=True)
print(f"Model loaded in {time.perf_counter() - t0:.2f} s")

vocab = sorted(set(k for kws in keyword_lists for k in kws))
print("Unique keywords:", len(vocab))
assert len(vocab) == 56635, f"expected 56635 keywords, got {len(vocab)}"
t0 = time.perf_counter()
embedding_matrix = model.encode(vocab, batch_size=32, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
print(f"Embeddings computed in {time.perf_counter() - t0:.1f} s (hardware dependent)")
embeddings = {kw: embedding_matrix[i] for i, kw in enumerate(vocab)}

t0 = time.perf_counter()
graph = ref.build_crs_for_tau(keyword_lists, embeddings, e3.TAU)
print(f"CRS built in {time.perf_counter() - t0:.1f} s | nodes: {graph.number_of_nodes()} | edges: {graph.number_of_edges()}")

result = e3.metrics(graph)
result["n_documents"] = len(corpus)
gate = e3.full_reference_gate(result)
print("\nReproduction gate (stored outputs of notebooks 2, 3, 4 and 5 at tau = 0.40):")
print(pd.DataFrame(gate["checks"])[["metric", "expected", "actual", "passed"]].to_string(index=False))
if not gate["passed"]:
    raise RuntimeError("STOPPED: the rebuilt CRS does not pass the full-corpus reference gate")
print("\nREFERENCE GATE: PASSED")

backbone = ref.build_backbone(graph, e3.BACKBONE)
assert backbone.number_of_nodes() == 408 and backbone.number_of_edges() == 608
partition = community_louvain.best_partition(backbone, weight="weight", random_state=e3.LOUVAIN_SEED)
modularity = community_louvain.modularity(partition, backbone, weight="weight")
assert len(set(partition.values())) == 7
assert abs(modularity - 0.3560443171946043) < 1e-6

df_comm = (pd.DataFrame({"keyword": list(partition.keys()), "community": list(partition.values())})
           .sort_values(["community", "keyword"]).reset_index(drop=True))
df_comm.to_csv(OUT / "crs_keyword_communities.csv", index=False)
df_edges = (pd.DataFrame([{"source": u, "target": v, "weight": d.get("weight"), "sim_count": d.get("sim_count"),
                           "sim_mean": d.get("sim_mean")} for u, v, d in backbone.edges(data=True)])
            .sort_values(["source", "target"]).reset_index(drop=True))
df_edges.to_csv(OUT / "crs_backbone_edges.csv", index=False)
df_sizes = (df_comm.groupby("community").size().rename("n_keywords").reset_index()
            .sort_values("n_keywords", ascending=False))
df_sizes.to_csv(OUT / "crs_community_sizes.csv", index=False)

print(f"\nBackbone: {backbone.number_of_nodes()} nodes, {backbone.number_of_edges()} edges | "
      f"communities: {len(set(partition.values()))} | modularity: {modularity:.10f}")
print("\nCommunity sizes (backbone concepts):")
print(df_sizes.to_string(index=False))
del embeddings, embedding_matrix, model
print(f"\nStage A: {(time.perf_counter() - stage_started) / 60:.1f} min")

Analysable documents: 52946
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | revision: e8f8c211226b | tau: 0.4 | backbone w >= 20 | Louvain seed: 42


Model loaded in 0.79 s
Unique keywords: 56635


Embeddings computed in 95.6 s (hardware dependent)


CRS built in 10.4 s | nodes: 56635 | edges: 109022



Reproduction gate (stored outputs of notebooks 2, 3, 4 and 5 at tau = 0.40):
                       metric      expected        actual  passed
                  n_documents  52946.000000  52946.000000    True
             nodes_full_graph  56635.000000  56635.000000    True
             edges_full_graph 109022.000000 109022.000000    True
      n_components_full_graph  19856.000000  19856.000000    True
                    lcc_nodes  34316.000000  34316.000000    True
                    lcc_edges 106297.000000 106297.000000    True
           lcc_fraction_nodes      0.605915      0.605915    True
           density_full_graph      0.000068      0.000068    True
               backbone_nodes    408.000000    408.000000    True
               backbone_edges    608.000000    608.000000    True
          backbone_components      2.000000      2.000000    True
           backbone_lcc_nodes    406.000000    406.000000    True
           backbone_lcc_edges    607.000000    607.000000    Tru

In [3]:
# ============================================================
# STAGE B1: MATCHED DATASET (TITLE + ABSTRACT OF THE 52,922 DOCUMENTS)
# ============================================================
# The record file is the input of notebook 1: one record per line, six fields joined by " • "
# (authors, title, year, source, abstract, original keywords). data/insumo_row_to_eid.csv links each
# record row to the Scopus EID of its published keyword list (one row per EID, record order).
records = pd.read_csv(PRIVATE_RECORDS)["insumo"].astype(str).tolist()
align_all = pd.read_csv(ALIGNMENT)
align = C.load_alignment(ALIGNMENT)
print(f"Record rows: {len(records):,} | alignment rows: {len(align_all):,} | rows matching a published output: "
      f"{int(align_all.matches_published.sum()):,} | unique EIDs: {len(align):,}")

main_eids = set(align_all[align_all.insumo_row <= RECORD_MAIN_BLOCK_LAST_ROW].eid.dropna())
tail = align_all[align_all.insumo_row > RECORD_MAIN_BLOCK_LAST_ROW]
print(f"Appended tail (rows > {RECORD_MAIN_BLOCK_LAST_ROW:,}): {len(tail)} rows | duplicates of main-block EIDs: "
      f"{int((tail.matches_published & tail.eid.isin(main_eids)).sum())} | rows without a published output: "
      f"{int((~tail.matches_published).sum())} | EIDs that appear only in the tail: "
      f"{int((tail.matches_published & ~tail.eid.isin(main_eids)).sum())}")

main_block = align[align.insumo_row <= RECORD_MAIN_BLOCK_LAST_ROW].reset_index(drop=True)
published = pd.read_excel(KEYWORDS)
published_keywords = dict(zip(published.EID_o_identificador.astype(str), published.palabras_clave))

rows = []
for r in main_block.itertuples(index=False):
    rec = C.parse_record(records[r.insumo_row])
    title, abstract = rec["title"], rec["abstract"]
    text_lda = f"{title} {abstract}".strip() if title else abstract   # title + abstract; abstract only if the title is missing
    rows.append({"insumo_row": int(r.insumo_row), "EID_clean": str(r.eid).strip(), "text_lda": text_lda,
                 "keywords_llm": published_keywords[str(r.eid)], "title_chars": len(title), "abstract_chars": len(abstract)})
df = pd.DataFrame(rows)
assert len(df) == 52922, f"expected 52922 matched documents, got {len(df)}"
assert df["EID_clean"].is_unique
print(f"\nMatched documents: {len(df):,} | empty titles: {int((df.title_chars == 0).sum())} | empty abstracts: "
      f"{int((df.abstract_chars == 0).sum())} | abstract field equal to the Scopus placeholder '[No abstract available]': "
      f"{int(df.text_lda.str.contains('No abstract available', regex=False).sum()):,}")
lengths = df.text_lda.str.len()
print(f"Text length in characters: median {int(lengths.median())}, min {int(lengths.min())}, max {int(lengths.max())}")

if ARCHIVED_ASSIGNMENTS is not None:
    same_set = set(df.EID_clean) == set(ARCHIVED_ASSIGNMENTS.EID_clean)
    same_order = df.EID_clean.tolist() == ARCHIVED_ASSIGNMENTS.EID_clean.tolist()
    print(f"EID set identical to the archived crs_document_assignments.csv: {same_set} (same order: {same_order})")
    assert same_set, "the matched document set differs from the archived run"

df[["insumo_row", "EID_clean", "text_lda", "keywords_llm"]].to_pickle(OUT / "e8_matched_dataset.pkl")   # gitignored: licensed text
print("Written (gitignored):", OUT / "e8_matched_dataset.pkl")

Record rows: 53,130 | alignment rows: 53,130 | rows matching a published output: 53,045 | unique EIDs: 52,947
Appended tail (rows > 52,922): 207 rows | duplicates of main-block EIDs: 98 | rows without a published output: 84 | EIDs that appear only in the tail: 25



Matched documents: 52,922 | empty titles: 0 | empty abstracts: 0 | abstract field equal to the Scopus placeholder '[No abstract available]': 0
Text length in characters: median 1331, min 15, max 15121
EID set identical to the archived crs_document_assignments.csv: True (same order: True)
Written (gitignored): results/e8_topic_modeling/e8_matched_dataset.pkl


In [4]:
# ============================================================
# STAGE B2: DOCUMENT-TERM MATRIX AND LDA FOR K IN {5, 10, 15, 20, 30, 40, 50}
# ============================================================
# Configuration recorded in the archived lda_config.json: CountVectorizer with English stop words,
# unigrams and bigrams, min_df 10, max_df 0.8, 30,000 features; LatentDirichletAllocation with batch
# variational inference, 50 iterations, random_state 42; K selected by the mean NPMI of the ten top
# terms per topic, perplexity kept as a secondary diagnostic.
t0 = time.perf_counter()
vectorizer = CountVectorizer(max_features=LDA["max_features"], min_df=LDA["min_df"], max_df=LDA["max_df"],
                             ngram_range=LDA["ngram_range"], stop_words=LDA["stop_words"])
X = vectorizer.fit_transform(df["text_lda"].fillna("").astype(str))
vectorization_seconds = time.perf_counter() - t0
feature_names = vectorizer.get_feature_names_out()
print(f"Document-term matrix: {X.shape} | nnz = {X.nnz:,} | bigrams in the vocabulary: "
      f"{int(sum(' ' in f for f in feature_names)):,} | {vectorization_seconds:.1f} s")
assert X.shape == (52922, 30000)
joblib.dump(vectorizer, OUT / "lda_vectorizer.joblib")

# NPMI of a topic: mean over the 45 pairs of its ten top terms of log(P(wi,wj) / (P(wi) P(wj))) / -log P(wi,wj),
# with probabilities estimated as document frequencies in the binarized document-term matrix
# (a pair that never co-occurs contributes -1).
X_binary = (X > 0).astype(np.float64).tocsc()
n_docs = X.shape[0]
document_frequency = np.asarray(X_binary.sum(axis=0)).ravel()

def topic_npmi(top_indices):
    sub = X_binary[:, top_indices]
    co_document = (sub.T @ sub).toarray()
    p_word = document_frequency[top_indices] / n_docs
    values = []
    for i in range(len(top_indices)):
        for j in range(i + 1, len(top_indices)):
            p_pair = co_document[i, j] / n_docs
            if p_pair == 0:
                values.append(-1.0)
            else:
                values.append(float(np.log(p_pair / (p_word[i] * p_word[j])) / -np.log(p_pair)))
    return float(np.mean(values))

selection_rows, topic_rows = [], []
fitting_started = time.perf_counter()
for k in KS:
    t0 = time.perf_counter()
    lda = LatentDirichletAllocation(n_components=k, max_iter=LDA["max_iter"], learning_method=LDA["learning_method"],
                                    random_state=LDA["random_state"])
    lda.fit(X)
    fit_seconds = time.perf_counter() - t0
    perplexity = float(lda.perplexity(X))
    npmis = []
    for topic_id, weights in enumerate(lda.components_):
        top = np.argsort(weights)[::-1][:LDA["top_words_for_npmi"]]
        npmi = topic_npmi(top)
        npmis.append(npmi)
        topic_rows.append({"K": k, "topic": topic_id, "npmi": npmi, "top_words": [str(feature_names[i]) for i in top]})
    selection_rows.append({"K": k, "NPMI": float(np.mean(npmis)), "perplexity": perplexity, "fit_time_seconds": fit_seconds})
    joblib.dump(lda, OUT / f"lda_k{k}.joblib")
    print(f"K={k:2d}: NPMI={np.mean(npmis):.6f} | perplexity={perplexity:.2f} | fit {fit_seconds / 60:.1f} min "
          f"(hardware dependent)", flush=True)
print(f"LDA fitting, all K: {(time.perf_counter() - fitting_started) / 60:.1f} min")

selection = pd.DataFrame(selection_rows)
selection["selected_by_npmi"] = selection["NPMI"] == selection["NPMI"].max()
selected_K = int(selection.loc[selection.selected_by_npmi, "K"].iloc[0])
topics = pd.DataFrame(topic_rows)
topics["top_words"] = topics["top_words"].apply(str)
topics.to_csv(OUT / "lda_topics_all_k.csv", index=False)
selection.to_csv(OUT / "lda_model_selection.csv", index=False)
lda_config = {"documents": int(len(df)), "text_input": "Title + Abstract; Abstract only if Title missing",
              "author_keywords_used": False, "index_keywords_used": False, "llm_keywords_used_as_lda_input": False,
              "K_values": KS, "selection_primary": "mean topic NPMI", "perplexity_role": "secondary diagnostic",
              "max_features": LDA["max_features"], "min_df": LDA["min_df"], "max_df": LDA["max_df"],
              "ngram_range": list(LDA["ngram_range"]), "stop_words": LDA["stop_words"], "max_iter": LDA["max_iter"],
              "learning_method": LDA["learning_method"], "random_state": LDA["random_state"],
              "top_words_for_npmi": LDA["top_words_for_npmi"], "vocabulary_size": int(len(feature_names)),
              "document_term_matrix_shape": list(X.shape), "vectorization_seconds": vectorization_seconds,
              "selected_K": selected_K}
(OUT / "lda_config.json").write_text(json.dumps(lda_config, indent=2, ensure_ascii=False) + "\n")

print("\nModel selection (K chosen by the highest mean NPMI; perplexity is a secondary diagnostic):")
print(selection.to_string(index=False))
print(f"\nSelected K = {selected_K}")
print(f"\nTen top terms of the first ten topics of the {selected_K}-topic model:")
print(topics[topics.K == selected_K].head(10)[["topic", "npmi", "top_words"]].to_string(index=False))

Document-term matrix: (52922, 30000) | nnz = 5,616,822 | bigrams in the vocabulary: 19,857 | 18.0 s


K= 5: NPMI=0.127083 | perplexity=4082.27 | fit 7.5 min (hardware dependent)


K=10: NPMI=0.113856 | perplexity=3737.84 | fit 8.5 min (hardware dependent)


K=15: NPMI=0.112625 | perplexity=3539.99 | fit 8.8 min (hardware dependent)


K=20: NPMI=0.134828 | perplexity=3441.50 | fit 8.8 min (hardware dependent)


K=30: NPMI=0.135669 | perplexity=3348.19 | fit 9.7 min (hardware dependent)


K=40: NPMI=0.141420 | perplexity=3281.98 | fit 9.9 min (hardware dependent)


K=50: NPMI=0.147862 | perplexity=3257.98 | fit 11.0 min (hardware dependent)


LDA fitting, all K: 65.6 min

Model selection (K chosen by the highest mean NPMI; perplexity is a secondary diagnostic):
 K     NPMI  perplexity  fit_time_seconds  selected_by_npmi
 5 0.127083 4082.267921        450.688231             False
10 0.113856 3737.841094        510.517246             False
15 0.112625 3539.994640        525.866608             False
20 0.134828 3441.499382        526.758154             False
30 0.135669 3348.188591        579.390287             False
40 0.141420 3281.977946        594.031672             False
50 0.147862 3257.976683        657.117947              True

Selected K = 50

Ten top terms of the first ten topics of the 50-topic model:
 topic     npmi                                                                                                                                          top_words
     0 0.101012                    ['systems', 'modelling', 'based', 'control', 'management', 'simulation', 'fuzzy', 'network', 'model', 'mathematical modell

In [5]:
# ============================================================
# STAGE B3: DOMINANT LDA TOPIC VS CRS MAJORITY COMMUNITY (NMI, ARI)
# ============================================================
comm = pd.read_csv(OUT / "crs_keyword_communities.csv")
assert len(df) == 52922 and df["EID_clean"].is_unique
assert len(comm) == 408 and comm["keyword"].is_unique and comm["community"].nunique() == 7
keyword_to_community = dict(zip(comm["keyword"], comm["community"]))

def parse_keywords(value):
    """Normalisation of the original comparison script: strip, lower, per-document deduplication."""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    elif pd.isna(value):
        values = []
    elif isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            values = list(parsed) if isinstance(parsed, (list, tuple, set)) else [value]
        except Exception:
            values = [value]
    else:
        values = []
    return sorted({str(x).strip().lower() for x in values if str(x).strip()})

def assign_crs(value):
    """Community holding a simple majority of the document's backbone keywords; ties are ambiguous."""
    kws = parse_keywords(value)
    mapped = [keyword_to_community[k] for k in kws if k in keyword_to_community]
    if not mapped:
        return pd.Series({"crs_status": "unassigned", "crs_community": np.nan,
                          "crs_mapped_keywords": 0, "crs_total_keywords": len(kws)})
    counts = Counter(mapped)
    max_count = max(counts.values())
    winners = [community for community, count in counts.items() if count == max_count]
    if len(winners) > 1:
        return pd.Series({"crs_status": "ambiguous", "crs_community": np.nan,
                          "crs_mapped_keywords": len(mapped), "crs_total_keywords": len(kws)})
    return pd.Series({"crs_status": "assigned", "crs_community": int(winners[0]),
                      "crs_mapped_keywords": len(mapped), "crs_total_keywords": len(kws)})

t0 = time.perf_counter()
assignments = df["keywords_llm"].apply(assign_crs)
df = pd.concat([df, assignments], axis=1)
print(f"CRS assignment completed in {time.perf_counter() - t0:.1f} s")

n_total = len(df)
n_assigned = int((df["crs_status"] == "assigned").sum())
n_ambiguous = int((df["crs_status"] == "ambiguous").sum())
n_unassigned = int((df["crs_status"] == "unassigned").sum())
coverage = n_assigned / n_total
print("\nCRS status of the documents:")
print(f"  Total:       {n_total:,}")
print(f"  Assigned:    {n_assigned:,} ({n_assigned / n_total:.4%})")
print(f"  Ambiguous:   {n_ambiguous:,} ({n_ambiguous / n_total:.4%})")
print(f"  Unassigned:  {n_unassigned:,} ({n_unassigned / n_total:.4%})")

# The original script reloaded the saved vectorizer and transformed the texts; the result is the fitted matrix.
t0 = time.perf_counter()
X_compare = joblib.load(OUT / "lda_vectorizer.joblib").transform(df["text_lda"].fillna("").astype(str))
assert X_compare.shape == (52922, 30000) and (X_compare != X).nnz == 0
print(f"\nLDA matrix: {X_compare.shape} | nnz = {X_compare.nnz:,} | {time.perf_counter() - t0:.1f} s")

mask = df["crs_status"].eq("assigned")
y_crs = df.loc[mask, "crs_community"].astype(int).to_numpy()

results = []
for k in KS:
    t0 = time.perf_counter()
    model_k = joblib.load(OUT / f"lda_k{k}.joblib")
    theta = model_k.transform(X_compare)
    dominant_topic = np.argmax(theta, axis=1)
    dominant_probability = np.max(theta, axis=1)
    df[f"lda_topic_k{k}"] = dominant_topic
    df[f"lda_probability_k{k}"] = dominant_probability
    y_lda = dominant_topic[mask.to_numpy()]
    nmi = normalized_mutual_info_score(y_crs, y_lda)
    ari = adjusted_rand_score(y_crs, y_lda)
    n_topics_observed = int(np.unique(y_lda).size)
    results.append({"K": k, "n_documents_total": n_total, "n_crs_assigned": n_assigned, "n_crs_ambiguous": n_ambiguous,
                    "n_crs_unassigned": n_unassigned, "crs_assignment_coverage": coverage,
                    "n_documents_compared": len(y_crs), "n_crs_communities_observed": int(np.unique(y_crs).size),
                    "n_lda_topics_observed": n_topics_observed, "NMI": nmi, "ARI": ari,
                    "mean_dominant_topic_probability_all": float(dominant_probability.mean()),
                    "mean_dominant_topic_probability_compared": float(dominant_probability[mask.to_numpy()].mean()),
                    "runtime_transform_seconds": time.perf_counter() - t0})
    print(f"  K={k:2d}: NMI={nmi:.6f} | ARI={ari:.6f} | topics observed={n_topics_observed} | "
          f"{results[-1]['runtime_transform_seconds']:.1f} s", flush=True)

comparison = pd.DataFrame(results)
doc_columns = ["EID_clean", "crs_status", "crs_community", "crs_mapped_keywords", "crs_total_keywords"]
for k in KS:
    doc_columns.extend([f"lda_topic_k{k}", f"lda_probability_k{k}"])
doc_output = df[doc_columns].copy()
doc_output.to_csv(OUT / "crs_document_assignments.csv", index=False)
comparison.to_csv(OUT / "lda_crs_comparison.csv", index=False)

crs_distribution = (df.loc[df["crs_status"] == "assigned", "crs_community"].astype(int).value_counts().sort_index()
                    .rename_axis("community").reset_index(name="n_documents"))
crs_distribution["proportion_assigned"] = crs_distribution["n_documents"] / n_assigned
crs_distribution.to_csv(OUT / "crs_document_community_distribution.csv", index=False)

print("\nLDA vs CRS agreement on the documents with an unambiguous community:")
print(comparison[["K", "n_documents_compared", "crs_assignment_coverage", "NMI", "ARI",
                  "mean_dominant_topic_probability_all"]].to_string(index=False))
print("\nDocuments per CRS community (assigned documents):")
print(crs_distribution.to_string(index=False))
print(f"\nThe selection of K is independent of the CRS: K = {selected_K} was selected by the highest NPMI among the values evaluated.")

CRS assignment completed in 7.3 s

CRS status of the documents:
  Total:       52,922
  Assigned:    39,456 (74.5550%)
  Ambiguous:   9,016 (17.0364%)
  Unassigned:  4,450 (8.4086%)



LDA matrix: (52922, 30000) | nnz = 5,616,822 | 6.5 s


  K= 5: NMI=0.143928 | ARI=0.094069 | topics observed=5 | 7.0 s


  K=10: NMI=0.119872 | ARI=0.069084 | topics observed=10 | 7.7 s


  K=15: NMI=0.122569 | ARI=0.056291 | topics observed=15 | 7.9 s


  K=20: NMI=0.115575 | ARI=0.046751 | topics observed=20 | 7.7 s


  K=30: NMI=0.098988 | ARI=0.031149 | topics observed=30 | 8.1 s


  K=40: NMI=0.104818 | ARI=0.030486 | topics observed=40 | 8.1 s


  K=50: NMI=0.100177 | ARI=0.026783 | topics observed=50 | 8.7 s



LDA vs CRS agreement on the documents with an unambiguous community:
 K  n_documents_compared  crs_assignment_coverage      NMI      ARI  mean_dominant_topic_probability_all
 5                 39456                  0.74555 0.143928 0.094069                             0.653210
10                 39456                  0.74555 0.119872 0.069084                             0.550656
15                 39456                  0.74555 0.122569 0.056291                             0.496355
20                 39456                  0.74555 0.115575 0.046751                             0.463101
30                 39456                  0.74555 0.098988 0.031149                             0.391865
40                 39456                  0.74555 0.104818 0.030486                             0.365550
50                 39456                  0.74555 0.100177 0.026783                             0.341596

Documents per CRS community (assigned documents):
 community  n_documents  proportion_ass

In [6]:
# ============================================================
# STAGE C: STRUCTURAL INTERPRETATION OF THE CRS COMMUNITIES VS LDA (K = 50)
# ============================================================
doc_assign = pd.read_csv(OUT / "crs_document_assignments.csv")
communities = pd.read_csv(OUT / "crs_keyword_communities.csv")
edges = pd.read_csv(OUT / "crs_backbone_edges.csv")
topics_all = pd.read_csv(OUT / "lda_topics_all_k.csv")
print(f"Documents: {len(doc_assign):,} | backbone concepts: {len(communities):,} | backbone edges: {len(edges):,} | "
      f"topic rows: {len(topics_all):,}")

# Degree and weighted degree of every concept inside the backbone
degree, weighted_degree = Counter(), Counter()
for _, row in edges.iterrows():
    u, v, w = row["source"], row["target"], float(row["weight"])
    degree[u] += 1
    degree[v] += 1
    weighted_degree[u] += w
    weighted_degree[v] += w
communities["degree_backbone"] = communities["keyword"].map(degree).fillna(0).astype(int)
communities["weighted_degree_backbone"] = communities["keyword"].map(weighted_degree).fillna(0.0)
communities = communities.sort_values(["community", "weighted_degree_backbone", "degree_backbone"],
                                      ascending=[True, False, False])
communities.to_csv(OUT / "crs_community_concept_metrics.csv", index=False)

# Community summaries: top concepts by weighted degree and the LDA-50 topics of their documents
assigned_docs = doc_assign[doc_assign["crs_status"] == "assigned"].copy()
assigned_docs["crs_community"] = assigned_docs["crs_community"].astype(int)
assigned_total = len(assigned_docs)
community_summary_rows, community_concept_rows = [], []
for community_id in sorted(communities["community"].unique()):
    subset_nodes = communities[communities["community"] == community_id].copy()
    subset_docs = assigned_docs[assigned_docs["crs_community"] == community_id].copy()
    top_nodes = subset_nodes.head(TOP_N_CONCEPTS)
    for rank, (_, row) in enumerate(top_nodes.iterrows(), start=1):
        community_concept_rows.append({"community": int(community_id), "rank": rank, "keyword": row["keyword"],
                                       "degree_backbone": row["degree_backbone"],
                                       "weighted_degree_backbone": row["weighted_degree_backbone"]})
    topic_counts = subset_docs["lda_topic_k50"].value_counts().head(TOP_N_TOPICS)
    top_topics_string = "; ".join([f"T{int(topic)}: {int(count)} ({count / len(subset_docs):.2%})"
                                   for topic, count in topic_counts.items()]) if len(subset_docs) else ""
    community_summary_rows.append({"community": int(community_id), "n_backbone_concepts": len(subset_nodes),
                                   "n_assigned_documents": len(subset_docs),
                                   "proportion_of_assigned_documents": len(subset_docs) / assigned_total,
                                   "top_backbone_concepts": "; ".join(top_nodes["keyword"].astype(str).tolist()),
                                   "top_lda50_topics": top_topics_string})
community_summary = pd.DataFrame(community_summary_rows)
community_concepts = pd.DataFrame(community_concept_rows)
community_summary.to_csv(OUT / "crs_lda50_community_summary.csv", index=False)
community_concepts.to_csv(OUT / "crs_top_concepts_by_community.csv", index=False)

# CRS community x LDA-50 topic cross-tabulation (counts, row proportions, long format)
cross = pd.crosstab(assigned_docs["crs_community"], assigned_docs["lda_topic_k50"])
cross.to_csv(OUT / "crs_lda50_crosstab_counts.csv")
cross_prop = cross.div(cross.sum(axis=1), axis=0)
cross_prop.to_csv(OUT / "crs_lda50_crosstab_row_proportions.csv")
long_rows = []
for community_id in cross.index:
    total = cross.loc[community_id].sum()
    for topic_id in cross.columns:
        count = int(cross.loc[community_id, topic_id])
        if count > 0:
            long_rows.append({"crs_community": int(community_id), "lda_topic_k50": int(topic_id),
                              "n_documents": count, "proportion_within_crs_community": count / total})
pd.DataFrame(long_rows).to_csv(OUT / "crs_lda50_overlap_long.csv", index=False)

# Inter-community edges of the backbone, pair summary and bridge concepts
keyword_to_comm = dict(zip(communities["keyword"], communities["community"]))
bridge_rows = []
for _, row in edges.iterrows():
    u, v = row["source"], row["target"]
    cu, cv = keyword_to_comm.get(u), keyword_to_comm.get(v)
    if cu is None or cv is None:
        continue
    if cu != cv:
        bridge_rows.append({"source": u, "source_community": int(cu), "target": v, "target_community": int(cv),
                            "weight": float(row["weight"])})
bridges = pd.DataFrame(bridge_rows)
if len(bridges):
    bridges = bridges.sort_values("weight", ascending=False)
bridges.to_csv(OUT / "crs_intercommunity_edges.csv", index=False)

if len(bridges):
    pair_summary = (bridges.assign(community_a=lambda x: x[["source_community", "target_community"]].min(axis=1),
                                   community_b=lambda x: x[["source_community", "target_community"]].max(axis=1))
                    .groupby(["community_a", "community_b"], as_index=False)
                    .agg(n_intercommunity_edges=("weight", "size"), total_intercommunity_weight=("weight", "sum"),
                         mean_intercommunity_weight=("weight", "mean"), max_intercommunity_weight=("weight", "max"))
                    .sort_values(["total_intercommunity_weight", "n_intercommunity_edges"], ascending=False))
else:
    pair_summary = pd.DataFrame()
pair_summary.to_csv(OUT / "crs_intercommunity_pair_summary.csv", index=False)

bridge_concept_stats, bridge_concept_weight = Counter(), Counter()
for _, row in bridges.iterrows():
    for node in [row["source"], row["target"]]:
        bridge_concept_stats[node] += 1
        bridge_concept_weight[node] += row["weight"]
bridge_concepts = pd.DataFrame([{"keyword": node, "community": int(keyword_to_comm[node]),
                                 "n_intercommunity_edges": n_edges, "intercommunity_weight": bridge_concept_weight[node],
                                 "degree_backbone": degree[node], "weighted_degree_backbone": weighted_degree[node]}
                                for node, n_edges in bridge_concept_stats.items()])
if len(bridge_concepts):
    bridge_concepts = bridge_concepts.sort_values(["intercommunity_weight", "n_intercommunity_edges"], ascending=False)
bridge_concepts.to_csv(OUT / "crs_bridge_concepts.csv", index=False)

print("\nMAIN COMMUNITIES")
for c in MAIN_COMMUNITIES:
    row = community_summary[community_summary["community"] == c]
    if row.empty:
        continue
    row = row.iloc[0]
    print(f"\nCRS community {c}: {row['n_backbone_concepts']} backbone concepts, {row['n_assigned_documents']:,} assigned "
          f"documents ({row['proportion_of_assigned_documents']:.2%})")
    print("  Top concepts:", row["top_backbone_concepts"])
    print("  Top LDA-50 topics:", row["top_lda50_topics"])
print(f"\nINTER-COMMUNITY EDGES OF THE BACKBONE: {len(bridges):,}")
print("\nCommunity pairs:")
print(pair_summary.to_string(index=False))
print(f"\nTop {TOP_N_BRIDGES} inter-community edges:")
print(bridges.head(TOP_N_BRIDGES).to_string(index=False))
print(f"\nTop {TOP_N_BRIDGES} bridge concepts:")
print(bridge_concepts.head(TOP_N_BRIDGES).to_string(index=False))

Documents: 52,922 | backbone concepts: 408 | backbone edges: 608 | topic rows: 170

MAIN COMMUNITIES

CRS community 0: 274 backbone concepts, 24,553 assigned documents (62.23%)
  Top concepts: mathematics education; teacher education; teacher training; preservice teachers; pedagogy; academic performance; academic achievement; primary education; teacher preparation; geometry; teacher knowledge; teaching methods; secondary education; student achievement; prospective teachers
  Top LDA-50 topics: T19: 2785 (11.34%); T49: 2441 (9.94%); T14: 1759 (7.16%); T28: 1239 (5.05%); T31: 1049 (4.27%)

CRS community 1: 42 backbone concepts, 4,377 assigned documents (11.09%)
  Top concepts: education; mathematics; engineering; science; curriculum; students; learning; technology; teaching; calculus; stem; numeracy; physics; teachers; research
  Top LDA-50 topics: T25: 430 (9.82%); T19: 408 (9.32%); T44: 222 (5.07%); T14: 209 (4.77%); T35: 187 (4.27%)

CRS community 2: 72 backbone concepts, 9,358 assign

## Comparison with the archived results

In [7]:
# ============================================================
# NEW OUTPUTS VS THE ARCHIVED RUN
# ============================================================
def load_new(name):
    return pd.read_csv(OUT / name)

rows = []
def report(table, quantity, value, note=""):
    rows.append({"file": table, "quantity": quantity, "result": value, "note": note})

if not ARCHIVED:
    print("No archived tables were present before this execution; nothing to compare.")
else:
    # CRS partition: same communities (as a partition) and same backbone edges with the same weights
    a, b = ARCHIVED["crs_keyword_communities.csv"], load_new("crs_keyword_communities.csv")
    m = a.merge(b, on="keyword", how="outer", suffixes=("_macos", "_new"))
    same_partition = (len(m) == len(a) == len(b) and m.isna().sum().sum() == 0
                      and normalized_mutual_info_score(m.community_macos, m.community_new) == 1.0)
    report("crs_keyword_communities.csv", "same 408 concepts and same partition (NMI = 1)", same_partition,
           "identical community labels" if same_partition and (m.community_macos == m.community_new).all() else "labels relabelled" if same_partition else "")
    a, b = ARCHIVED["crs_backbone_edges.csv"], load_new("crs_backbone_edges.csv")
    key = lambda d: set(zip(d.apply(lambda r: tuple(sorted((r.source, r.target))), axis=1), d.weight))
    report("crs_backbone_edges.csv", "same 608 edges with the same support", key(a) == key(b) and len(a) == len(b) == 608)
    a, b = ARCHIVED["crs_community_sizes.csv"], load_new("crs_community_sizes.csv")
    report("crs_community_sizes.csv", "same community sizes", sorted(a.n_keywords) == sorted(b.n_keywords))

    # LDA model selection
    a, b = ARCHIVED["lda_model_selection.csv"], load_new("lda_model_selection.csv")
    m = a.merge(b, on="K", suffixes=("_macos", "_new"))
    for col in ["NPMI", "perplexity"]:
        d = (m[f"{col}_macos"] - m[f"{col}_new"]).abs()
        report("lda_model_selection.csv", f"max |difference| in {col} over K", float(d.max()),
               "; ".join(f"K={k}: {x:.6f} vs {y:.6f}" for k, x, y in zip(m.K, m[f"{col}_macos"], m[f"{col}_new"])) if col == "NPMI" else "")
    report("lda_model_selection.csv", "same selected K", bool(m.loc[m.selected_by_npmi_macos, "K"].tolist() == m.loc[m.selected_by_npmi_new, "K"].tolist()),
           f"archived {m.loc[m.selected_by_npmi_macos, 'K'].tolist()}, new {m.loc[m.selected_by_npmi_new, 'K'].tolist()}")
    report("lda_model_selection.csv", "fit_time_seconds", "not compared", "hardware dependent")

    # Topic coherence per topic and top-term overlap
    a, b = ARCHIVED["lda_topics_all_k.csv"], load_new("lda_topics_all_k.csv")
    m = a.merge(b, on=["K", "topic"], suffixes=("_macos", "_new"))
    jac = m.apply(lambda r: len(set(ast.literal_eval(r.top_words_macos)) & set(ast.literal_eval(r.top_words_new))) /
                  len(set(ast.literal_eval(r.top_words_macos)) | set(ast.literal_eval(r.top_words_new))), axis=1)
    report("lda_topics_all_k.csv", "max |difference| in per-topic NPMI", float((m.npmi_macos - m.npmi_new).abs().max()))
    report("lda_topics_all_k.csv", "mean Jaccard of the ten top terms per (K, topic)", round(float(jac.mean()), 4),
           f"{int((jac == 1).sum())} of {len(jac)} topics with identical top-term sets")

    # Agreement measures and assignment counts
    a, b = ARCHIVED["lda_crs_comparison.csv"], load_new("lda_crs_comparison.csv")
    m = a.merge(b, on="K", suffixes=("_macos", "_new"))
    for col in ["n_crs_assigned", "n_crs_ambiguous", "n_crs_unassigned", "n_documents_compared", "n_lda_topics_observed"]:
        report("lda_crs_comparison.csv", f"{col} identical for every K", bool((m[f"{col}_macos"] == m[f"{col}_new"]).all()))
    for col in ["NMI", "ARI", "mean_dominant_topic_probability_all"]:
        d = (m[f"{col}_macos"] - m[f"{col}_new"]).abs()
        report("lda_crs_comparison.csv", f"max |difference| in {col} over K", float(d.max()),
               "; ".join(f"K={k}: {x:.4f} vs {y:.4f}" for k, x, y in zip(m.K, m[f"{col}_macos"], m[f"{col}_new"])) if col in ("NMI", "ARI") else "")
    report("lda_crs_comparison.csv", "runtime_transform_seconds", "not compared", "hardware dependent")

    # Document-level assignments: CRS status and community, LDA dominant topic per K
    if ARCHIVED_ASSIGNMENTS is not None:
        new_assign = load_new("crs_document_assignments.csv")
        m = ARCHIVED_ASSIGNMENTS.merge(new_assign, on="EID_clean", suffixes=("_macos", "_new"))
        report("crs_document_assignments.csv", "CRS status identical for every document", bool((m.crs_status_macos == m.crs_status_new).all()))
        both = m.crs_status_macos.eq("assigned") & m.crs_status_new.eq("assigned")
        report("crs_document_assignments.csv", "CRS community identical for every assigned document",
               bool((m.loc[both, "crs_community_macos"] == m.loc[both, "crs_community_new"]).all()))
        for k in KS:
            share = float((m[f"lda_topic_k{k}_macos"] == m[f"lda_topic_k{k}_new"]).mean())
            nmi_k = normalized_mutual_info_score(m[f"lda_topic_k{k}_macos"], m[f"lda_topic_k{k}_new"])
            report("crs_document_assignments.csv", f"dominant LDA topic K={k}: share of documents with the same topic id",
                   round(share, 4), f"NMI between archived and new topic assignments {nmi_k:.4f} (labels may be permuted)")

    # Community-level tables
    a, b = ARCHIVED["crs_document_community_distribution.csv"], load_new("crs_document_community_distribution.csv")
    report("crs_document_community_distribution.csv", "same documents per community", sorted(a.n_documents) == sorted(b.n_documents))
    a, b = ARCHIVED["crs_lda50_community_summary.csv"], load_new("crs_lda50_community_summary.csv")
    m = a.merge(b, on="community", suffixes=("_macos", "_new"))
    report("crs_lda50_community_summary.csv", "same concept and document counts per community",
           bool((m.n_backbone_concepts_macos == m.n_backbone_concepts_new).all() and (m.n_assigned_documents_macos == m.n_assigned_documents_new).all()))
    report("crs_lda50_community_summary.csv", "same top backbone concepts per community", bool((m.top_backbone_concepts_macos == m.top_backbone_concepts_new).all()))
    share_top = lambda s: float(s.split("(")[1].split("%")[0]) if isinstance(s, str) and "(" in s else np.nan
    report("crs_lda50_community_summary.csv", "max |difference| in the share of the most frequent LDA-50 topic (percentage points)",
           round(float((m.top_lda50_topics_macos.map(share_top) - m.top_lda50_topics_new.map(share_top)).abs().max()), 2),
           "topic ids may be permuted between fits; the share of the most frequent topic is compared")
    a, b = ARCHIVED["crs_intercommunity_pair_summary.csv"], load_new("crs_intercommunity_pair_summary.csv")
    report("crs_intercommunity_pair_summary.csv", "same (edges, total weight) multiset over community pairs",
           sorted(zip(a.n_intercommunity_edges, a.total_intercommunity_weight)) == sorted(zip(b.n_intercommunity_edges, b.total_intercommunity_weight)))
    a, b = ARCHIVED["crs_intercommunity_edges.csv"], load_new("crs_intercommunity_edges.csv")
    report("crs_intercommunity_edges.csv", "same inter-community edges with the same support", key(a) == key(b) and len(a) == len(b))
    a, b = ARCHIVED["crs_bridge_concepts.csv"], load_new("crs_bridge_concepts.csv")
    m = a.merge(b, on="keyword", how="outer", suffixes=("_macos", "_new"))
    report("crs_bridge_concepts.csv", "same bridge concepts with the same edge counts and weights",
           bool(len(m) == len(a) == len(b) and (m.n_intercommunity_edges_macos == m.n_intercommunity_edges_new).all()
                and (m.intercommunity_weight_macos == m.intercommunity_weight_new).all()))
    if ARCHIVED_CONFIG is not None:
        new_config = json.loads((OUT / "lda_config.json").read_text())
        diff_keys = [k for k in ARCHIVED_CONFIG if k != "vectorization_seconds" and ARCHIVED_CONFIG[k] != new_config.get(k)]
        report("lda_config.json", "keys other than vectorization_seconds identical", not diff_keys, ", ".join(diff_keys))

    archive_comparison = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 200)
    print(archive_comparison.to_string(index=False))

                                   file                                                                            quantity       result                                                                                                                                                                                              note
            crs_keyword_communities.csv                                      same 408 concepts and same partition (NMI = 1)         True                                                                                                                                                                        identical community labels
                 crs_backbone_edges.csv                                                same 608 edges with the same support         True                                                                                                                                                                                                  
       

## Check against the manuscript

Communities are identified by content (the ones holding *mathematics education*, *stem education* and *artificial intelligence*), since Louvain labels are arbitrary.

In [8]:
# ============================================================
# CHECK AGAINST THE MANUSCRIPT (values transcribed from main.tex)
# ============================================================
def fmt_like(value, template):
    """Format value with the precision (decimals, thousands separator) of the manuscript string."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "--"
    decimals = len(template.split(".")[1]) if "." in template else 0
    return f"{value:,.{decimals}f}" if "," in template else f"{value:.{decimals}f}"

check = []
def add(location, quantity, manuscript, value, note=""):
    computed = fmt_like(value, manuscript) if not isinstance(value, str) else value
    check.append({"location": location, "quantity": quantity, "manuscript": manuscript, "computed": computed,
                  "flag": "match" if computed == manuscript else "differs", "note": note})

selection = pd.read_csv(OUT / "lda_model_selection.csv")
comparison = pd.read_csv(OUT / "lda_crs_comparison.csv")
summary_c = pd.read_csv(OUT / "crs_lda50_community_summary.csv")
pairs = pd.read_csv(OUT / "crs_intercommunity_pair_summary.csv")
inter = pd.read_csv(OUT / "crs_intercommunity_edges.csv")
bridge = pd.read_csv(OUT / "crs_bridge_concepts.csv")
comm_of = dict(zip(pd.read_csv(OUT / "crs_keyword_communities.csv").keyword, pd.read_csv(OUT / "crs_keyword_communities.csv").community))
c_math, c_stem, c_ai = comm_of["mathematics education"], comm_of["stem education"], comm_of["artificial intelligence"]
sel = comparison.merge(selection, on="K")
k_sel = int(selection.loc[selection.selected_by_npmi, "K"].iloc[0])
row_sel = sel[sel.K == k_sel].iloc[0]

add("Abstract", "NMI between CRS communities and LDA topics (selected K)", "0.09", float(row_sel.NMI), f"K = {k_sel}")
add("Sec. IV-D3", "documents matched by identifier", "52,922", float(comparison.n_documents_total.iloc[0]))
add("Sec. IV-D3", "LDA features (unigrams and bigrams)", "30,000", float(json.loads((OUT / "lda_config.json").read_text())["vocabulary_size"]))
add("Sec. IV-D3", "K with the highest NPMI", "50", float(k_sel))
add("Sec. IV-D3", "NPMI at the selected K", "0.157", float(row_sel.NPMI))
add("Sec. IV-D3", "NPMI increases with K (monotone over the seven values)", "increased with K",
    "increased with K" if selection.sort_values("K").NPMI.is_monotonic_increasing else "not monotone",
    "; ".join(f"K={k}: {v:.3f}" for k, v in zip(selection.K, selection.NPMI)))
add("Sec. IV-D3", "documents with an unambiguous community", "39,456", float(row_sel.n_crs_assigned))
add("Sec. IV-D3", "share with an unambiguous community", "74.6%", f"{100 * row_sel.n_crs_assigned / row_sel.n_documents_total:.1f}%")
add("Sec. IV-D3", "documents with a tie", "9,016", float(row_sel.n_crs_ambiguous))
add("Sec. IV-D3", "share with a tie", "17.0%", f"{100 * row_sel.n_crs_ambiguous / row_sel.n_documents_total:.1f}%")
add("Sec. IV-D3", "documents with no keyword in the backbone", "4,450", float(row_sel.n_crs_unassigned))
add("Sec. IV-D3", "share with no keyword in the backbone", "8.4%", f"{100 * row_sel.n_crs_unassigned / row_sel.n_documents_total:.1f}%")
add("Sec. IV-D3", "documents on which agreement is computed", "39,456", float(row_sel.n_documents_compared))

TABLE9 = {5: ("0.110", "0.131", "0.116"), 10: ("0.119", "0.134", "0.074"), 15: ("0.132", "0.116", "0.041"),
          20: ("0.126", "0.108", "0.035"), 30: ("0.139", "0.099", "0.034"), 40: ("0.147", "0.098", "0.026"),
          50: ("0.157", "0.095", "0.023")}
for k, (npmi_s, nmi_s, ari_s) in TABLE9.items():
    r = sel[sel.K == k].iloc[0]
    add(f"Table 9, K = {k}", "NPMI", npmi_s, float(r.NPMI))
    add(f"Table 9, K = {k}", "NMI", nmi_s, float(r.NMI))
    add(f"Table 9, K = {k}", "ARI", ari_s, float(r.ARI))
add("Sec. IV-D3", "NMI range over K (min)", "0.095", float(comparison.NMI.min()))
add("Sec. IV-D3", "NMI range over K (max)", "0.134", float(comparison.NMI.max()))
add("Sec. IV-D3", "ARI range over K (min)", "0.023", float(comparison.ARI.min()))
add("Sec. IV-D3", "ARI range over K (max)", "0.116", float(comparison.ARI.max()))
add("Sec. IV-D3", "K with the lowest NMI and the lowest ARI", "50",
    f"{int(comparison.loc[comparison.NMI.idxmin(), 'K'])} / {int(comparison.loc[comparison.ARI.idxmin(), 'K'])}".replace("50 / 50", "50"))

def community_row(cid):
    return summary_c[summary_c.community == cid].iloc[0]
def top_share(cid):
    return float(community_row(cid).top_lda50_topics.split("(")[1].split("%")[0])
add("Sec. IV-D3", "largest community: backbone concepts", "274", float(community_row(c_math).n_backbone_concepts), "community of mathematics education")
add("Sec. IV-D3", "largest community: documents", "24,553", float(community_row(c_math).n_assigned_documents))
add("Sec. IV-D3", "largest community: share of its most frequent LDA-50 topic", "8.4%", f"{top_share(c_math):.1f}%")
add("Sec. IV-D3", "second community: backbone concepts", "72", float(community_row(c_stem).n_backbone_concepts), "community of stem education")
add("Sec. IV-D3", "second community: documents", "9,358", float(community_row(c_stem).n_assigned_documents))
add("Sec. IV-D3", "second community: share of its most frequent LDA-50 topic", "16.7%", f"{top_share(c_stem):.1f}%")
add("Sec. IV-D3", "computing community: backbone concepts", "14", float(community_row(c_ai).n_backbone_concepts), "community of artificial intelligence")
add("Sec. IV-D3", "computing community: documents", "1,041", float(community_row(c_ai).n_assigned_documents))
add("Sec. IV-D3", "computing community: share of its most frequent LDA-50 topic", "41.9%", f"{top_share(c_ai):.1f}%")
ai_members = set(pd.read_csv(OUT / "crs_keyword_communities.csv").query("community == @c_ai").keyword)
add("Sec. IV-D3", "computing community holds artificial intelligence, computational thinking, programming, machine learning", "yes",
    "yes" if {"artificial intelligence", "computational thinking", "programming", "machine learning"} <= ai_members else "no")

add("Sec. IV-D3", "inter-community edges of the backbone", "104", float(len(inter)))
pair = pairs[(pairs.community_a == min(c_math, c_stem)) & (pairs.community_b == max(c_math, c_stem))].iloc[0]
add("Sec. IV-D3", "edges between the mathematics-education and the STEM communities", "66", float(pair.n_intercommunity_edges))
add("Sec. IV-D3", "aggregate support of those edges (documents)", "5,480", float(pair.total_intercommunity_weight))
def support(a, b):
    r = inter[((inter.source == a) & (inter.target == b)) | ((inter.source == b) & (inter.target == a))]
    return float(r.weight.iloc[0]) if len(r) else float("nan")
for other, s in [("science education", "722"), ("stem education", "415"), ("higher education", "384"),
                 ("engineering education", "333"), ("computational thinking", "209"), ("artificial intelligence", "208")]:
    add("Sec. IV-D3", f"support of mathematics education with {other}", s, support("mathematics education", other))
def bridge_row(kw):
    return bridge[bridge.keyword == kw].iloc[0]
add("Sec. IV-D3", "inter-community edges involving mathematics education", "44", float(bridge_row("mathematics education").n_intercommunity_edges))
add("Sec. IV-D3", "aggregate weight of those edges", "5,819", float(bridge_row("mathematics education").intercommunity_weight))
add("Sec. IV-D3", "inter-community edges involving stem education", "24", float(bridge_row("stem education").n_intercommunity_edges))
add("Sec. IV-D3", "inter-community edges involving science education", "7", float(bridge_row("science education").n_intercommunity_edges))

manuscript_check = pd.DataFrame(check)
pd.set_option("display.max_colwidth", 100)
print(manuscript_check.to_string(index=False))
n_match = int((manuscript_check.flag == "match").sum())
print(f"\n{n_match} of {len(manuscript_check)} rows match; {len(manuscript_check) - n_match} differ.")
if n_match < len(manuscript_check):
    print("\nDiffering rows:")
    print(manuscript_check[manuscript_check.flag != "match"].to_string(index=False))

       location                                                                                                 quantity       manuscript     computed    flag                                                                                     note
       Abstract                                                  NMI between CRS communities and LDA topics (selected K)             0.09         0.10 differs                                                                                   K = 50
     Sec. IV-D3                                                                          documents matched by identifier           52,922       52,922   match                                                                                         
     Sec. IV-D3                                                                      LDA features (unigrams and bigrams)           30,000       30,000   match                                                                                         
     Sec